# DAViD Sketch Notebook

This notebook orchestrates the training pipeline. All logic lives in:
- `config.py`: paths, hyperparameters
- `dataset.py`: `OptimizedVideoDataset`, data loaders
- `model.py`: `ClassificationHead`, `SwiGLU`, transforms
- `encoder.py`: ViFi-CLIP feature extractor loader
- `train.py`: training loop, validation, seed setting
- `clip/`: vendored CLIP architecture (for weight compatibility)

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!git clone https://github.com/aitf-its-tim3-dfk/david.git

In [ ]:
%cd david
%pip install ftfy regex decord scikit-learn

## 2. Load Feature Extractor

In [ ]:
from config import CLIP_ARCH, CLASS_NAMES, VIFICLIP_CHECKPOINT
from encoder import load_feature_extractor

feature_extractor = load_feature_extractor(
    arch=CLIP_ARCH,
    class_names=CLASS_NAMES,
    checkpoint_path=VIFICLIP_CHECKPOINT,
).cuda()

## 3. Prepare Data

In [ ]:
import os
from config import METADATA_CSV, BASE_DIR, BATCH_SIZE, VAL_SPLIT, NUM_WORKERS
from config import TRAIN_SIZE, VAL_SIZE, BALANCE, NUM_FRAMES, CLEAN_CSV, OUTPUT_DIR
from config import INPUT_DIM, NUM_CLASSES, LEARNING_RATE, NUM_EPOCHS, BEST_MODEL_SAVE_PATH, HEAD_TYPE
from config import MIN_SEG_SECS, MAX_SEG_SECS

In [ ]:
from model import build_transform
from dataset import get_train_val_loaders
from train import set_seed

set_seed(42)

preprocess = build_transform(224)
train_loader, val_loader, val_files = get_train_val_loaders(
    transform=preprocess,
    csv_path=METADATA_CSV,
    base_dir=BASE_DIR,
    val_split=VAL_SPLIT,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    num_frames=NUM_FRAMES,
    validate=True,
    clean_csv_path=CLEAN_CSV,
    balance=BALANCE,
    min_seg_secs=MIN_SEG_SECS,
    max_seg_secs=MAX_SEG_SECS,
)

## 3b. Prepare Data — Legacy (Google Drive + JSON cache)

**Skip this section if you already ran section 3 above.**

Use this if your dataset is still in the old Google Drive directory structure (no `metadata.csv`).
This mode scans directories, validates each video with `decord`, and caches the file list to a
local JSON file for faster subsequent runs.

In [ ]:
import os, shutil

# ── Legacy paths (pre-refactor Google Drive structure) ────────────────────────
LEGACY_BASE_DIR   = "/content/drive/MyDrive/data mining gemastik"
LEGACY_CACHE_PATH = "video_train_10000_cache_fixed_2.json"
LEGACY_DRIVE_CACHE = (
    "/content/drive/MyDrive/data mining gemastik/"
    "video train 10000/video_train_10000_cache_fixed_2.json"
)

LEGACY_REAL_DIRS = [
    "/content/drive/.shortcut-targets-by-id/"
    "1Wcbv564DV62urzCJvYmvnkDD_Z74ZKLa/GenVideo-Val/Real",
    os.path.join(LEGACY_BASE_DIR, "K4/videos_val"),
]
LEGACY_FAKE_DIRS = [
    "/content/drive/MyDrive/gemastik-datmin/pika/train_pika",
    os.path.join(LEGACY_BASE_DIR, "Sora/train_OpenSora"),
    os.path.join(LEGACY_BASE_DIR, "SVD/train_SVD"),
]

# Copy cache locally for faster reads (skip if already exists or not on Drive)
if not os.path.exists(LEGACY_CACHE_PATH) and os.path.exists(LEGACY_DRIVE_CACHE):
    shutil.copy(LEGACY_DRIVE_CACHE, LEGACY_CACHE_PATH)
    print(f"Copied cache to {LEGACY_CACHE_PATH}")
elif os.path.exists(LEGACY_CACHE_PATH):
    print(f"Cache already exists at {LEGACY_CACHE_PATH}")

In [ ]:
from model import build_transform
from dataset import get_train_val_loaders
from config import BATCH_SIZE, VAL_SPLIT, NUM_WORKERS, TRAIN_SIZE, VAL_SIZE

preprocess = build_transform(224)
train_loader, val_loader, val_files = get_train_val_loaders(
    transform=preprocess,
    # No csv_path — triggers legacy JSON cache + directory scan mode
    real_dirs=LEGACY_REAL_DIRS,
    fake_dirs=LEGACY_FAKE_DIRS,
    cache_path=LEGACY_CACHE_PATH,
    val_split=VAL_SPLIT,
    train_size=TRAIN_SIZE,
    val_size=VAL_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

## 4. Train

In [ ]:
from config import USE_AMP, LR_SCHEDULER, PATIENCE
from config import WANDB_ENABLED, WANDB_PROJECT, WANDB_RUN_NAME
from config import CLIP_ARCH, CLASS_NAMES
from train import set_seed, run_training

set_seed(42)

classifier = run_training(
    feature_extractor=feature_extractor,
    train_loader=train_loader,
    val_loader=val_loader,
    input_dim=INPUT_DIM,
    num_classes=NUM_CLASSES,
    head_type=HEAD_TYPE,
    lr=LEARNING_RATE,
    num_epochs=NUM_EPOCHS,
    save_path=BEST_MODEL_SAVE_PATH,
    use_amp=USE_AMP,
    lr_scheduler=LR_SCHEDULER,
    patience=PATIENCE,
    use_wandb=WANDB_ENABLED,
    wandb_project=WANDB_PROJECT,
    wandb_run_name=WANDB_RUN_NAME,
    wandb_extra_config={
        "clip_arch":   CLIP_ARCH,
        "class_names": list(CLASS_NAMES),
        "num_frames":  NUM_FRAMES,
        "balance":     BALANCE,
    },
)

In [ ]:
from evaluate import evaluate
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
metrics = evaluate(
    classifier, feature_extractor, val_loader, val_files, device,
    use_wandb=WANDB_ENABLED,
)

## 6. Save & Cleanup

In [ ]:
## 6. Save & Cleanup

In [ ]:
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()